#Manual Ensemble Tool
[ZFTurbo](https://github.com/ZFTurbo/Music-Source-Separation-Training) / [jarredou](https://github.com/jarredou/Music-Source-Separation-Training-Colab-Inference/)

In [ ]:
#@markdown # Install
from google.colab import output
import subprocess, threading, time

def spinner(stop_event):
    frames = "|/-\\"
    i = 0
    while not stop_event.is_set():
        print(f"\rInstalling... {frames[i % len(frames)]}", end="")
        time.sleep(0.1)
        i += 1
    print("\r", end="")

def install(commands, cwd=None):
    stop = threading.Event()
    t = threading.Thread(target=spinner, args=(stop,))
    t.start()

    try:
        for cmd in commands:
            subprocess.check_output(
                cmd,
                shell=True,
                cwd=cwd,
                stderr=subprocess.STDOUT,
            )

        stop.set()
        t.join()
        output.clear()
        print("✅ Install successful")

    except subprocess.CalledProcessError as e:
        stop.set()
        t.join()
        print("\n❌ Install failed\n")
        print(f"Command: {cmd}\n")
        print(e.output.decode())
        raise



#install
install(
    [
        #"pip install -q librosa soundfile",
        "wget https://raw.githubusercontent.com/jarredou/Music-Source-Separation-Training-Colab-Inference/refs/heads/main/ensemble.py",
    ],
    cwd="/content",
)

In [ ]:
#@markdown #Gdrive connection
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@markdown #Ensemble
output_file = "/content/output.wav" #@param {type:"string"}

#@markdown <font size=2>*Documentation about the different types: https://github.com/ZFTurbo/Music-Source-Separation-Training/blob/main/docs/ensemble.md*
type = "avg_wave" #@param ["avg_wave", "median_wave", "min_wave", "max_wave", "avg_fft", "median_fft", "min_fft", "max_fft"]
n_fft = 4096 #@param [8192, 4096, 2048, 1024] {type:"raw"}

trim_to_shortest = False #@param {type:"boolean"}
_trim_to_shortest = '--trim_to_shortest' if trim_to_shortest is True else ''
#@markdown ---
#@markdown *Leave fields empty if you don't use them all !*

input_file_1 = "" #@param {type:"string"}
input_file_2 = "" #@param {type:"string"}
input_file_3 = "" #@param {type:"string"}
input_file_4 = "" #@param {type:"string"}
input_file_5 = "" #@param {type:"string"}
input_file_6 = "" #@param {type:"string"}
input_file_7 = "" #@param {type:"string"}
input_file_8 = "" #@param {type:"string"}
input_file_9 = "" #@param {type:"string"}
input_file_10 = "" #@param {type:"string"}

weight_file_1 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_2 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_3 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_4 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_5 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_6 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_7 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_8 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_9 = 1 #@param {type:"slider", min:1, max:10, step:1}
weight_file_10 = 1 #@param {type:"slider", min:1, max:10, step:1}


input_files = []
for i in range(1, 11):
    input_files.append(globals()[f'input_file_{i}'])
# print(input_files)

weights = []
for i in range(1, 11):
    weights.append(globals()[f'weight_file_{i}'])
# print(weights)

# remove empty inputs and add quotes
input_files = [f'"{file}"' for file in input_files if file]

total_input = len(input_files)
# print(f'number of input files = {total_input}')

input_files_concat = ' '.join(input_files)

# number of weight entries must be same as number of input files
weights = [w for w in weights][:total_input]

weights_concat = ' '.join(map(str, weights))
%cd '/content'

!python ensemble.py \
    --files {input_files_concat} \
    --weights {weights_concat} \
    --type {type} \
    --output {output_file} \
    --n_fft {n_fft} \
    --hop_length {n_fft // 4} \
    {_trim_to_shortest}